In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('../data/raw/rainfall_india.csv',low_memory=False)
df.shape

(1308174, 25)

In [3]:
df.head()

,Country,State,District,Year,Calendar Day,Obj_Id,Daily Category,Weekly Date,Weekly Category,Cumulative Date,...,"Percentage Of Daily Departure (UOM:%(Percentage)), Scaling Factor:1","Weekly Actual (UOM:mm(Millimeter)), Scaling Factor:1","Weekly Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Weekly Departure (UOM:%(Percentage)), Scaling Factor:1","Cumulative Actual (UOM:mm(Millimeter)), Scaling Factor:1","Cumulative Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Cumulative Departure (UOM:%(Percentage)), Scaling Factor:1","Monthly Acutual (UOM:mm(Millimeter)), Scaling Factor:1","Monthly Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Monthly Departure (UOM:%(Percentage)), Scaling Factor:1"
0,India,Andaman and Nicobar Islands,Nicobars,"Calendar Year (Jan - Dec), 2026",2026-09-12,573.0,D,03-09-2026 To 09-09-2026,D,2026-06-01,...,-22.0,7.6,9.8,-22.0,497.7,896.7,-44.0,31.8,105.4,-70.0
1,India,Andaman and Nicobar Islands,North and Middle Andaman,"Calendar Year (Jan - Dec), 2026",2026-09-12,571.0,E,03-09-2026 To 09-09-2026,E,2026-06-01,...,25.0,18.7,15.0,25.0,2357.4,1628.3,45.0,416.9,197.5,111.0
2,India,Andaman and Nicobar Islands,South Andamans,"Calendar Year (Jan - Dec), 2026",2026-09-12,572.0,NR,03-09-2026 To 09-09-2026,NR,2026-06-01,...,-100.0,0.0,12.3,-100.0,1119.5,1368.6,-18.0,110.3,162.1,-32.0
3,India,Andhra Pradesh,Alluri Sitharama Raju,"Calendar Year (Jan - Dec), 2026",2026-09-12,735.0,N,03-09-2026 To 09-09-2026,N,2026-06-01,...,17.0,11.2,9.6,17.0,385.8,791.6,-51.0,23.6,85.8,-72.0
4,India,Andhra Pradesh,Anakapalli,"Calendar Year (Jan - Dec), 2026",2026-09-12,736.0,N,03-09-2026 To 09-09-2026,N,2026-06-01,...,9.0,10.6,9.7,9.0,318.0,520.4,-39.0,52.0,71.8,-28.0


In [4]:
### Remove duplicates rows ####

df.isnull().sum()


Country                                                                          0
State                                                                            0
District                                                                         0
Year                                                                             0
Calendar Day                                                                     0
Obj_Id                                                                      787174
Daily Category                                                                   0
Weekly Date                                                                 793149
Weekly Category                                                             787175
Cumulative Date                                                             788464
Cumulative Category                                                         787174
Monthly Date                                                                     0
Mont

In [5]:
df = df.drop_duplicates().copy()
len(df)

1281331

In [6]:
###  Clean State and District ###
df['State'] = df['State'].astype(str).str.strip()
df['District'] = df['District'].astype(str).str.strip()


In [7]:
### Convert Date ###
df["date"] = pd.to_datetime(df['Calendar Day'],errors='coerce')
df["date"].min(),df["date"].max()

(Timestamp('2021-01-01 00:00:00'), Timestamp('2026-09-12 00:00:00'))

In [8]:
df["date"].isnull().sum()

np.int64(0)

In [9]:
### Remove unusable records ###
df = df.dropna(
    subset=["State", "District", "date"]
).copy()

In [10]:
### Remove "Unknown Districts Of India" ###
mask = df["District"].str.contains(
    "Unknown Districts Of India",
    case=False,
    na=False
)

df = df[~mask].copy()

In [11]:
### Convert rainfall columns to numbers ###
actual_col = "Daily Actual (UOM:mm(Millimeter)), Scaling Factor:1"
normal_col = "Daily Normal (UOM:mm(Millimeter)), Scaling Factor:1"

df[actual_col] = pd.to_numeric(
    df[actual_col], errors="coerce"
)

df[normal_col] = pd.to_numeric(
    df[normal_col], errors="coerce"
)

In [12]:
### Create the analytical dataset ###
daily = (
    df.groupby(
        ["State", "District", "date"],
        as_index=False
    )
    .agg({
        actual_col: "mean",
        normal_col: "mean"
    })
)

In [13]:
df.head()

,Country,State,District,Year,Calendar Day,Obj_Id,Daily Category,Weekly Date,Weekly Category,Cumulative Date,...,"Weekly Actual (UOM:mm(Millimeter)), Scaling Factor:1","Weekly Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Weekly Departure (UOM:%(Percentage)), Scaling Factor:1","Cumulative Actual (UOM:mm(Millimeter)), Scaling Factor:1","Cumulative Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Cumulative Departure (UOM:%(Percentage)), Scaling Factor:1","Monthly Acutual (UOM:mm(Millimeter)), Scaling Factor:1","Monthly Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Monthly Departure (UOM:%(Percentage)), Scaling Factor:1",date
0,India,Andaman and Nicobar Islands,Nicobars,"Calendar Year (Jan - Dec), 2026",2026-09-12,573.0,D,03-09-2026 To 09-09-2026,D,2026-06-01,...,7.6,9.8,-22.0,497.7,896.7,-44.0,31.8,105.4,-70.0,2026-09-12
1,India,Andaman and Nicobar Islands,North and Middle Andaman,"Calendar Year (Jan - Dec), 2026",2026-09-12,571.0,E,03-09-2026 To 09-09-2026,E,2026-06-01,...,18.7,15.0,25.0,2357.4,1628.3,45.0,416.9,197.5,111.0,2026-09-12
2,India,Andaman and Nicobar Islands,South Andamans,"Calendar Year (Jan - Dec), 2026",2026-09-12,572.0,NR,03-09-2026 To 09-09-2026,NR,2026-06-01,...,0.0,12.3,-100.0,1119.5,1368.6,-18.0,110.3,162.1,-32.0,2026-09-12
3,India,Andhra Pradesh,Alluri Sitharama Raju,"Calendar Year (Jan - Dec), 2026",2026-09-12,735.0,N,03-09-2026 To 09-09-2026,N,2026-06-01,...,11.2,9.6,17.0,385.8,791.6,-51.0,23.6,85.8,-72.0,2026-09-12
4,India,Andhra Pradesh,Anakapalli,"Calendar Year (Jan - Dec), 2026",2026-09-12,736.0,N,03-09-2026 To 09-09-2026,N,2026-06-01,...,10.6,9.7,9.0,318.0,520.4,-39.0,52.0,71.8,-28.0,2026-09-12


In [14]:
daily.rename(
    columns={
        actual_col: "actual_mm",
        normal_col: "normal_mm"
    },
    inplace=True
)

In [15]:
daily.duplicated(
    ["State", "District", "date"]
).sum()

np.int64(0)

In [16]:
### Create important analytical features ###
daily["year"] = daily["date"].dt.year

In [17]:
daily["month"] = daily["date"].dt.month

In [18]:
daily["month_name"] = daily["date"].dt.month_name()

In [19]:
daily["season"] = np.select(
    [
        daily["month"].isin([1, 2]),
        daily["month"].isin([3, 4, 5]),
        daily["month"].isin([6, 7, 8, 9]),
        daily["month"].isin([10, 11, 12])
    ],
    [
        "Winter",
        "Pre-Monsoon",
        "Southwest Monsoon",
        "Post-Monsoon"
    ],
    default="Unknown"
)

In [20]:
daily["season"].value_counts()

season
Southwest Monsoon    452884
Pre-Monsoon          354898
Post-Monsoon         262154
Winter               190904
Name: count, dtype: int64

In [21]:
### Create rainfall anomaly ###
daily["anomaly_mm"] = (
    daily["actual_mm"] - daily["normal_mm"]
)

In [22]:
df.head()

,Country,State,District,Year,Calendar Day,Obj_Id,Daily Category,Weekly Date,Weekly Category,Cumulative Date,...,"Weekly Actual (UOM:mm(Millimeter)), Scaling Factor:1","Weekly Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Weekly Departure (UOM:%(Percentage)), Scaling Factor:1","Cumulative Actual (UOM:mm(Millimeter)), Scaling Factor:1","Cumulative Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Cumulative Departure (UOM:%(Percentage)), Scaling Factor:1","Monthly Acutual (UOM:mm(Millimeter)), Scaling Factor:1","Monthly Normal (UOM:mm(Millimeter)), Scaling Factor:1","Percentage Of Monthly Departure (UOM:%(Percentage)), Scaling Factor:1",date
0,India,Andaman and Nicobar Islands,Nicobars,"Calendar Year (Jan - Dec), 2026",2026-09-12,573.0,D,03-09-2026 To 09-09-2026,D,2026-06-01,...,7.6,9.8,-22.0,497.7,896.7,-44.0,31.8,105.4,-70.0,2026-09-12
1,India,Andaman and Nicobar Islands,North and Middle Andaman,"Calendar Year (Jan - Dec), 2026",2026-09-12,571.0,E,03-09-2026 To 09-09-2026,E,2026-06-01,...,18.7,15.0,25.0,2357.4,1628.3,45.0,416.9,197.5,111.0,2026-09-12
2,India,Andaman and Nicobar Islands,South Andamans,"Calendar Year (Jan - Dec), 2026",2026-09-12,572.0,NR,03-09-2026 To 09-09-2026,NR,2026-06-01,...,0.0,12.3,-100.0,1119.5,1368.6,-18.0,110.3,162.1,-32.0,2026-09-12
3,India,Andhra Pradesh,Alluri Sitharama Raju,"Calendar Year (Jan - Dec), 2026",2026-09-12,735.0,N,03-09-2026 To 09-09-2026,N,2026-06-01,...,11.2,9.6,17.0,385.8,791.6,-51.0,23.6,85.8,-72.0,2026-09-12
4,India,Andhra Pradesh,Anakapalli,"Calendar Year (Jan - Dec), 2026",2026-09-12,736.0,N,03-09-2026 To 09-09-2026,N,2026-06-01,...,10.6,9.7,9.0,318.0,520.4,-39.0,52.0,71.8,-28.0,2026-09-12


In [23]:
### Create rainfall flags ###
daily["rain_flag"] = (
    daily["actual_mm"] > 0
).astype(int)

In [24]:
daily["heavy_rain_flag"] = (
    daily["actual_mm"] >= 64.5
).astype(int)

In [25]:
daily["very_heavy_rain_flag"] = (
    daily["actual_mm"] >= 115.6
).astype(int)

In [26]:
daily["extreme_rain_flag"] = (
    daily["actual_mm"] >= 204.5
).astype(int)

In [27]:
daily["rainfall_intensity"] = np.select(
    [
        daily["actual_mm"].isna(),
        daily["actual_mm"] == 0,
        daily["actual_mm"].between(0.1, 2.4),
        daily["actual_mm"].between(2.5, 15.5),
        daily["actual_mm"].between(15.6, 64.4),
        daily["actual_mm"].between(64.5, 115.5),
        daily["actual_mm"].between(115.6, 204.4),
        daily["actual_mm"] >= 204.5
    ],
    [
        "Missing",
        "No Rain",
        "Very Light",
        "Light",
        "Moderate",
        "Heavy",
        "Very Heavy",
        "Extremely Heavy"
    ],
    default="Unknown"
)

In [28]:
### Final data check ###
daily.head()

,State,District,date,actual_mm,normal_mm,year,month,month_name,season,anomaly_mm,rain_flag,heavy_rain_flag,very_heavy_rain_flag,extreme_rain_flag,rainfall_intensity
0,Andaman and Nicobar Islands,Nicobars,2021-01-01,NaN,NaN,2021,1,January,Winter,NaN,0,0,0,0,Missing
1,Andaman and Nicobar Islands,Nicobars,2021-01-02,NaN,NaN,2021,1,January,Winter,NaN,0,0,0,0,Missing
2,Andaman and Nicobar Islands,Nicobars,2021-01-03,NaN,NaN,2021,1,January,Winter,NaN,0,0,0,0,Missing
3,Andaman and Nicobar Islands,Nicobars,2021-01-05,NaN,NaN,2021,1,January,Winter,NaN,0,0,0,0,Missing
4,Andaman and Nicobar Islands,Nicobars,2021-01-06,NaN,NaN,2021,1,January,Winter,NaN,0,0,0,0,Missing


In [29]:
daily.shape

(1260840, 15)

In [30]:
daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 1260840 entries, 0 to 1260839
Data columns (total 15 columns):
 #   Column                Non-Null Count    Dtype         
---  ------                --------------    -----         
 0   State                 1260840 non-null  str           
 1   District              1260840 non-null  str           
 2   date                  1260840 non-null  datetime64[us]
 3   actual_mm             1243768 non-null  float64       
 4   normal_mm             1255919 non-null  float64       
 5   year                  1260840 non-null  int32         
 6   month                 1260840 non-null  int32         
 7   month_name            1260840 non-null  str           
 8   season                1260840 non-null  str           
 9   anomaly_mm            1243767 non-null  float64       
 10  rain_flag             1260840 non-null  int64         
 11  heavy_rain_flag       1260840 non-null  int64         
 12  very_heavy_rain_flag  1260840 non-null  int64        

In [31]:
daily.isna().sum()

State                       0
District                    0
date                        0
actual_mm               17072
normal_mm                4921
year                        0
month                       0
month_name                  0
season                      0
anomaly_mm              17073
rain_flag                   0
heavy_rain_flag             0
very_heavy_rain_flag        0
extreme_rain_flag           0
rainfall_intensity          0
dtype: int64

In [32]:
daily.duplicated(
    ["State", "District", "date"]
).sum()

np.int64(0)

In [33]:
from pathlib import Path

Path("../data/processed").mkdir(
    parents=True,
    exist_ok=True
)

daily.to_csv(
    "../data/processed/rainfall_daily_clean.csv",
    index=False
)